In [1]:
pip install ipykernel

Note: you may need to restart the kernel to use updated packages.


In [2]:
import sys

print(sys.executable)

d:\ChatBot_Try\.venv\Scripts\python.exe


In [3]:
!pip install python-dotenv
!pip install -U langchain-huggingface huggingface_hub

In [4]:
!pip install langgraph

In [5]:
from dotenv import load_dotenv
from typing import TypedDict, Annotated
from langgraph.graph import StateGraph, START, END
from langchain_huggingface import HuggingFaceEndpoint, ChatHuggingFace
from langchain_core.messages import BaseMessage, HumanMessage
from langgraph.graph.message import add_messages
load_dotenv()

True

In [6]:
llm = HuggingFaceEndpoint(
    repo_id="openai/gpt-oss-20b",
    task="conversational",
    temperature=0.3,
    max_new_tokens=512,
)

model = ChatHuggingFace(llm=llm)

In [7]:
class Convo(TypedDict):

    messages: Annotated[list[BaseMessage], add_messages]

In [8]:
def run_model(state: Convo):

    response = model.invoke(state["messages"])
    return {'messages': [response]}

In [9]:
from langgraph.checkpoint.memory import InMemorySaver

In [11]:
graph = StateGraph(Convo)

graph.add_node('talking_phase', run_model)

graph.add_edge(START, 'talking_phase')
graph.add_edge('talking_phase', END)

from langgraph.checkpoint.postgres import PostgresSaver
import os

DATABASE_URL = os.getenv("DATABASE_URL")

checkpointer_context = PostgresSaver.from_conn_string(DATABASE_URL)

checkpointer = checkpointer_context.__enter__()

checkpointer.setup()

workflow = graph.compile(
    checkpointer=checkpointer
)

In [12]:
config1 = {"configurable": {"thread_id": "2"}}

In [13]:
result= workflow.invoke({'messages':[HumanMessage(content="Hi, What is my name?")]}, config=config1)
print(result)

{'messages': [HumanMessage(content='Hi, What is my name?', additional_kwargs={}, response_metadata={}, id='a901e8aa-e8c9-479f-a212-a50f0c823721'), AIMessage(content='I’m not sure what your name is—could you let me know?', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 75, 'prompt_tokens': 78, 'total_tokens': 153}, 'model_name': 'openai/gpt-oss-20b', 'system_fingerprint': 'fp_c5a89987dc', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--01a0cef6-54f1-7bd2-a673-9987663cc68b-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 78, 'output_tokens': 75, 'total_tokens': 153})]}


In [14]:
!pip install psycopg[binary] langgraph-checkpoint-postgres

In [15]:
import psycopg
import os

DATABASE_URL = os.getenv("DATABASE_URL")

print("Connected!")

Connected!


In [16]:
def create_thread(title="New Chat"):
    with psycopg.connect(DATABASE_URL) as conn:
        with conn.cursor() as cur:
            cur.execute(
                """
                INSERT INTO threads (title)
                VALUES (%s)
                RETURNING thread_id;
                """,
                (title,)
            )

            thread_id = cur.fetchone()[0]

    return str(thread_id)

In [17]:
def save_message(thread_id, role, content):
    with psycopg.connect(DATABASE_URL) as conn:
        with conn.cursor() as cur:
            cur.execute(
                """
                INSERT INTO messages (thread_id, role, content)
                VALUES (%s, %s, %s);
                """,
                (thread_id, role, content)
            )

In [22]:
#thread_id = create_thread("Chat with Aryan")
thread_id = str("3bcd2326-c553-44a8-9d8d-59c3d7f9e534")
config = {
    "configurable": {
        "thread_id": str("3bcd2326-c553-44a8-9d8d-59c3d7f9e534")
    }
}

In [23]:
user_message = "Hi, what is my name?"

save_message(
    thread_id,
    "user",
    user_message
)

result = workflow.invoke(
    {
        "messages": [
            HumanMessage(content=user_message)
        ]
    },
    config=config
)

assistant_message = result["messages"][-1].content

save_message(
    thread_id,
    "assistant",
    assistant_message
)

print(assistant_message)

I’m not sure what your name is—could you let me know?


In [ ]:
def get_messages(thread_id):
    with psycopg.connect(DATABASE_URL) as conn:
        with conn.cursor() as cur:
            cur.execute(
                """
                SELECT role, content, created_at
                FROM messages
                WHERE thread_id = %s
                ORDER BY created_at ASC;
                """,
                (thread_id,)
            )

            return cur.fetchall()

In [ ]:
def get_threads():
    with psycopg.connect(DATABASE_URL) as conn:
        with conn.cursor() as cur:
            cur.execute(
                """
                SELECT *
                FROM threads
                """
            )

            return cur.fetchall()

In [ ]:
messages = get_messages(thread_id)

for message in messages:
    print(message)

('user', 'Hi, what is my name?', datetime.datetime(2026, 9, 23, 15, 44, 21, 983691, tzinfo=datetime.timezone.utc))
('assistant', 'I’m not sure what your name is—could you let me know?', datetime.datetime(2026, 9, 23, 15, 44, 23, 729324, tzinfo=datetime.timezone.utc))


In [ ]:
print(get_threads())

[(UUID('3bcd2326-c553-44a8-9d8d-59c3d7f9e534'), 'Chat with Aryan', datetime.datetime(2026, 9, 23, 14, 32, 8, 635280, tzinfo=datetime.timezone.utc), datetime.datetime(2026, 9, 23, 14, 32, 8, 635280, tzinfo=datetime.timezone.utc)), (UUID('65e1cbfb-3bef-4355-b273-f9ede0d7d642'), 'Chat with Aryan', datetime.datetime(2026, 9, 23, 15, 44, 20, 584326, tzinfo=datetime.timezone.utc), datetime.datetime(2026, 9, 23, 15, 44, 20, 584326, tzinfo=datetime.timezone.utc))]
